## YouTube Scrape → Markdown

Scrape videos from the UVA RC YouTube playlist, fetch metadata and English transcripts, and save one Markdown file per video under `data/video/`.

Open WebUI handles chunking, embeddings, vector storage, and retrieval later.

## 1. Load Video List from Playlist

In [1]:
from yt_dlp import YoutubeDL

PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLT4bryHgBcRP7N-hB9u6EWs6tq_2nMoRO"

ydl_opts = {"extract_flat": True, "quiet": True}

with YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(PLAYLIST_URL, download=False)

entries = info.get("entries", [])

print(f"Found {len(entries)} videos in: {info.get('title', 'Unknown')}")

for entry in entries:
    print(f"  - {entry['title']} ({entry['url']})")

Found 4 videos in: RC Tutorial Series
  - Connecting to HPC (https://www.youtube.com/watch?v=BpaFQG4JOEU)
  - Open OnDemand Interactive Apps (https://www.youtube.com/watch?v=FUZbumfxGyY)
  - Features of Open OnDemand (https://www.youtube.com/watch?v=MpzThi43iak)
  - Working with Files (https://www.youtube.com/watch?v=asN63Ujhzks)


## 2. Fetch Metadata & Transcripts

Fetch each video's metadata and full English transcript. No manual chunking or LangChain `Document` objects are needed.

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi

def get_video_metadata(video_url):
    """Fetch video metadata using yt_dlp."""
    with YoutubeDL({"quiet": True, "skip_download": True}) as ydl:
        info = ydl.extract_info(video_url, download=False)

    return {
        "title": info.get("title", ""),
        "author": info.get("uploader", ""),
        "description": info.get("description", ""),
        "length": info.get("duration", 0),
        "publish_date": info.get("upload_date", ""),
        "webpage_url": info.get("webpage_url", video_url),
    }

def get_transcript(video_id):
    """Fetch the full English transcript for a video."""
    api = YouTubeTranscriptApi()
    transcript = api.fetch(video_id, languages=["en"])

    return " ".join(
        entry.text.strip()
        for entry in transcript
        if entry.text.strip()
    )

videos = []

for entry in entries:
    video_url = entry["url"]
    video_id = entry.get("id", video_url.split("v=")[-1])

    print(f"Loading: {entry['title']}")

    try:
        metadata = get_video_metadata(video_url)
        transcript = get_transcript(video_id)

        videos.append({
            "id": video_id,
            "metadata": metadata,
            "transcript": transcript,
        })

        print(f"  -> Transcript: {len(transcript)} characters")

    except Exception as e:
        print(f"  FAILED: {e}")

print(f"\nSuccessfully processed {len(videos)} videos.")

Loading: Connecting to HPC


  -> Transcript: 7670 characters
Loading: Open OnDemand Interactive Apps


  -> Transcript: 12927 characters
Loading: Features of Open OnDemand


  -> Transcript: 11466 characters
Loading: Working with Files


  -> Transcript: 11572 characters

Successfully processed 4 videos.


## 3. Preview

In [3]:
for video in videos[:3]:
    metadata = video["metadata"]

    print(f"=== {metadata['title']} ===")
    print(f"URL: {metadata['webpage_url']}")
    print(f"Author: {metadata['author']}")
    print(f"Transcript: {video['transcript'][:300]}...")
    print()

=== Connecting to HPC ===
URL: https://www.youtube.com/watch?v=BpaFQG4JOEU
Author: UVA Research Computing
Transcript: Narrator: Hello and welcome back to the University of Virginia's high-performance 
computing tutorial series. In this module, we will cover three different 
ways to connect to the HPC system at UVA. The first method is Open OnDemand, which is a 
web application accessed through a web browser. From t...

=== Open OnDemand Interactive Apps ===
URL: https://www.youtube.com/watch?v=FUZbumfxGyY
Author: UVA Research Computing
Transcript: Narrator: Hello and welcome back to 
the University of Virginia's High Performance Computing tutorial series. In this module, we will be covering the various interactive apps that you can 
access through Open OnDemand. These are all GUI apps like JupyterLab and 
RStudio Server that you can run direc...

=== Features of Open OnDemand ===
URL: https://www.youtube.com/watch?v=MpzThi43iak
Author: UVA Research Computing
Transcript: Narrator: Hello

## 4. Generate Markdown Files

Write one Markdown file per successfully processed video to `data/video/`.

In [4]:
from pathlib import Path
import re

# nbconvert executes this notebook from the scrapers directory.
# This also works if the notebook is opened from the repo root.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "scrapers" else cwd

OUTPUT_FOLDER = PROJECT_ROOT / "data" / "video"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

def safe_filename(filename):
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    filename = re.sub(r'\s+', ' ', filename).strip()
    return filename[:150]

created = 0
failed = 0

for video in videos:
    try:
        metadata = video["metadata"]
        transcript = video["transcript"]

        title = metadata["title"]
        filename = safe_filename(f"youtube_{title}.md")
        file_path = OUTPUT_FOLDER / filename

        markdown = f"""# {title}

## Video Information

**Source:** YouTube  
**Author:** {metadata['author']}  
**URL:** {metadata['webpage_url']}  
**Publish Date:** {metadata['publish_date']}

---

## Description

{metadata['description']}

---

## Transcript

{transcript}
"""

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(markdown.strip() + "\n")

        created += 1
        print(f"Created: {filename}")

    except Exception as e:
        failed += 1
        print(f"ERROR: {e}")

print("Video Markdown Generation Complete")
print(f"Created: {created}")
print(f"Failed:  {failed}")
print(f"Output folder: {OUTPUT_FOLDER}")

Created: youtube_Connecting to HPC.md
Created: youtube_Open OnDemand Interactive Apps.md
Created: youtube_Features of Open OnDemand.md
Created: youtube_Working with Files.md
Video Markdown Generation Complete
Created: 4
Failed:  0
Output folder: C:\Users\mayoe\OneDrive\Desktop\kb_Upload\rag-kb-upload\data\video
